# 02 - Sentinel-2 Vegetation Indices

**What:** Compute every built-in vegetation index that the RAVI plugin supports on a
Sentinel-2 image / collection, using Google Earth Engine. For each index we show the
formula, compute it with the *exact* expression and band aliasing from the legacy plugin,
and print the index description.

**Why:** Vegetation indices turn raw Sentinel-2 reflectance into agronomic information
(vigor, chlorophyll, water content, burn severity). This notebook is the runnable,
inspectable reference for the formulas the plugin ships.

**Legacy reference:**
- `legacy:ravi_dialog.py` -> `RaviDialog.calculate_vegetation_index(image, index_name)`
  (the per-index `ee.Image.expression` / `normalizedDifference` definitions and the
  Sentinel-2 band mapping with `/10000` scaling).
- `legacy:modules/vegetation_index_info.py` -> `vegetation_indices` (HTML descriptions/formulas).

**Indices covered:** NDVI, GNDVI, EVI, EVI2, SAVI (L=0.5), MSAVI, SFDVI, CIgreen, NDRE,
ARVI, NDMI, NBR, SIPI, NDWI, ReCI, MTCI, MCARI, VARI, TVI.

**Band aliasing (Sentinel-2 L2A, `COPERNICUS/S2_SR_HARMONIZED`):**

| Alias | Band | Description |
|-------|------|-------------|
| Blue | B2 | Blue |
| Green | B3 | Green |
| Red | B4 | Red |
| RedEdge1 | B5 | Red Edge 1 |
| NIR | B8 | Near-infrared |
| SWIR1 | B11 | Shortwave infrared 1 |
| SWIR2 | B12 | Shortwave infrared 2 |

> **Scaling:** Surface-reflectance bands are scaled by `/10000` in the ratio/expression
> indices (EVI, SAVI, ...), exactly as the legacy code does. The `normalizedDifference`
> indices (NDVI, GNDVI, NDRE, NDMI, NBR, NDWI) are scale-invariant, so legacy does **not**
> divide those. `CIgreen`, `ReCI`, `MTCI` legacy code uses **raw** (unscaled) bands -- this
> is preserved faithfully below.

In [ ]:
# Setup: Earth Engine + pandas (geemap optional)
import os
import ee
import pandas as pd

# Service-account auth via the GEE env var (same as dev/testing.ipynb).
credentials = ee.ServiceAccountCredentials(None, os.environ["GEE"])
ee.Initialize(credentials)

# Optional interactive map (guarded so the notebook runs without geemap installed).
try:
    import geemap
    HAS_GEEMAP = True
except Exception:
    HAS_GEEMAP = False
print("Earth Engine initialized. geemap available:", HAS_GEEMAP)

In [ ]:
import zipfile

import geopandas as gpd


def load_aoi_from_shapefile(shapefile_path):
    """Load an AOI from a (zipped) shapefile as an ee.FeatureCollection.

    Same loader as dev/testing.ipynb: reproject to EPSG:4326, dissolve to a
    single geometry, strip any Z coordinate, wrap in a FeatureCollection.
    """
    if shapefile_path.endswith(".zip"):
        with zipfile.ZipFile(shapefile_path, "r") as zip_ref:
            shapefile_within_zip = None
            for file in zip_ref.namelist():
                if file.lower().endswith(".shp"):
                    shapefile_within_zip = file
                    break
            if not shapefile_within_zip:
                raise FileNotFoundError(f"No .shp file found inside the zip archive: {shapefile_path}")
            gpd_aoi = gpd.read_file(f"zip://{shapefile_path}/{shapefile_within_zip}")
    else:
        gpd_aoi = gpd.read_file(shapefile_path)

    gpd_aoi = gpd_aoi.to_crs(epsg=4326)

    if gpd_aoi.empty:
        raise ValueError(f"The shapefile at {shapefile_path} does not contain any geometries.")

    if len(gpd_aoi) > 1:
        gpd_aoi = gpd_aoi.dissolve()

    geometry = gpd_aoi.geometry.iloc[0]
    geojson = geometry.__geo_interface__

    if geojson["type"] == "Polygon":
        geojson["coordinates"] = [
            list(map(lambda coord: coord[:2], ring)) for ring in geojson["coordinates"]
        ]
    elif geojson["type"] == "MultiPolygon":
        geojson["coordinates"] = [
            [list(map(lambda coord: coord[:2], ring)) for ring in polygon]
            for polygon in geojson["coordinates"]
        ]

    ee_geometry = ee.Geometry(geojson)
    feature = ee.Feature(ee_geometry)
    return ee.FeatureCollection([feature])


# AOI loaded from the project shapefile (same area as dev/testing.ipynb).
aoi = load_aoi_from_shapefile("contorno_area_total.zip").geometry()
start, end = "2023-01-01", "2024-01-01"

# Legacy uses COPERNICUS/S2_SR_HARMONIZED filtered by CLOUDY_PIXEL_PERCENTAGE,
# then an SCL-based cloud/shadow mask (legacy: RaviDialog.SCL_filter / SCL_mask).
# SCL classes removed here: 3 (cloud shadow), 8 (cloud medium prob),
# 9 (cloud high prob), 10 (thin cirrus).
def mask_s2_scl(image):
    scl = image.select("SCL")
    mask = ee.Image.constant(1)
    for class_value in (3, 8, 9, 10):
        mask = mask.And(scl.neq(class_value))
    return image.updateMask(mask)

collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi)
    .filterDate(start, end)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
    .map(mask_s2_scl)
)

print("Images in collection:", collection.size().getInfo())

# Median composite -> the single ee.Image we feed to every index function below.
image = collection.median().clip(aoi)

# Helper: mean of the "index" band over the AOI, so each cell is verifiable.
def index_mean(index_image, scale=10):
    return (
        index_image.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=aoi, scale=scale, maxPixels=1e13
        )
        .get("index")
        .getInfo()
    )

## NDVI

**Formula:** `NDVI = (NIR - RED) / (NIR + RED)`

Normalized Difference Vegetation Index. Classic vigor/greenness indicator from the contrast between NIR and red reflectance. Range -1..1; higher = denser, healthier vegetation.

*Legacy: `calculate_vegetation_index(image, "NDVI")`; description from `vegetation_indices["NDVI"]`.*

In [ ]:
def ndvi(image):
    # legacy: image.normalizedDifference(["B8", "B4"]).rename("index")
    return image.normalizedDifference(["B8", "B4"]).rename("index")

ndvi_img = ndvi(image)
print("NDVI mean over AOI:", index_mean(ndvi_img))

## GNDVI

**Formula:** `GNDVI = (NIR - GREEN) / (NIR + GREEN)`

Green NDVI. Uses the green band instead of red; more sensitive to canopy chlorophyll content. Useful for plant stress, nutrient deficiencies, crop vigor.

*Legacy: `calculate_vegetation_index(image, "GNDVI")`; description from `vegetation_indices["GNDVI"]`.*

In [ ]:
def gndvi(image):
    # legacy: image.normalizedDifference(["B8", "B3"]).rename("index")
    return image.normalizedDifference(["B8", "B3"]).rename("index")

gndvi_img = gndvi(image)
print("GNDVI mean over AOI:", index_mean(gndvi_img))

## EVI

**Formula:** `EVI = 2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))`

Enhanced Vegetation Index. Adds a blue-band aerosol correction and a canopy-background adjustment to reduce atmospheric/soil noise and limit NDVI saturation in high biomass.

*Legacy: `calculate_vegetation_index(image, "EVI")`; description from `vegetation_indices["EVI"]`.*

In [ ]:
def evi(image):
    # legacy: bands scaled by /10000
    nir = image.select("B8").divide(10000)
    red = image.select("B4").divide(10000)
    blue = image.select("B2").divide(10000)
    return image.expression(
        "2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))",
        {"NIR": nir, "RED": red, "BLUE": blue},
    ).rename("index")

evi_img = evi(image)
print("EVI mean over AOI:", index_mean(evi_img))

## EVI2

**Formula:** `EVI2 = 2.5 * ((NIR - RED) / (NIR + RED + 1))`

EVI without the blue band. Keeps most of EVI's high-biomass advantages while being simpler and usable on sensors lacking a reliable blue band.

*Legacy: `calculate_vegetation_index(image, "EVI2")`; description from `vegetation_indices["EVI2"]`.*

In [ ]:
def evi2(image):
    nir = image.select("B8").divide(10000)
    red = image.select("B4").divide(10000)
    return image.expression(
        "2.5 * ((NIR - RED) / (NIR + RED + 1))",
        {"NIR": nir, "RED": red},
    ).rename("index")

evi2_img = evi2(image)
print("EVI2 mean over AOI:", index_mean(evi2_img))

## SAVI

**Formula:** `SAVI = (1 + L) * ((NIR - RED) / (NIR + RED + L))   with L = 0.5`

Soil-Adjusted Vegetation Index. The L term suppresses soil-background reflectance; good for sparse vegetation / arid areas. Plugin fixes L = 0.5.

*Legacy: `calculate_vegetation_index(image, "SAVI")`; description from `vegetation_indices["SAVI"]`.*

In [ ]:
def savi(image):
    nir = image.select("B8").divide(10000)
    red = image.select("B4").divide(10000)
    L = 0.5  # legacy: soil brightness correction factor fixed at 0.5
    return image.expression(
        "(1 + L) * ((NIR - RED) / (NIR + RED + L))",
        {"NIR": nir, "RED": red, "L": L},
    ).rename("index")

savi_img = savi(image)
print("SAVI mean over AOI:", index_mean(savi_img))

## MSAVI

**Formula:** `MSAVI = (2*NIR + 1 - sqrt((2*NIR + 1)^2 - 8*(NIR - RED))) / 2`

Modified SAVI. Replaces SAVI's fixed L with a self-adjusting term driven by vegetation density, giving more consistent results across mixed cover without prior knowledge of L.

*Legacy: `calculate_vegetation_index(image, "MSAVI")`; description from `vegetation_indices["MSAVI"]`.*

In [ ]:
def msavi(image):
    nir = image.select("B8").divide(10000)
    red = image.select("B4").divide(10000)
    return image.expression(
        "((2 * NIR + 1) - sqrt((2 * NIR + 1) ** 2 - 8 * (NIR - RED))) / 2",
        {"NIR": nir, "RED": red},
    ).rename("index")

msavi_img = msavi(image)
print("MSAVI mean over AOI:", index_mean(msavi_img))

## SFDVI

**Formula:** `SFDVI = ((NIR + GREEN)/2 - (RED + REDEDGE)/2)`

Spectral Feature Depth Vegetation Index. Combines NIR/green vs red/red-edge to probe spectral feature depth; shows more gradation in dense vegetation than NDVI/RENDVI.

*Legacy: `calculate_vegetation_index(image, "SFDVI")`; description from `vegetation_indices["SFDVI"]`.*

In [ ]:
def sfdvi(image):
    # legacy: REDEDGE = B5
    return image.expression(
        "((NIR + GREEN)/2 - (RED + REDEDGE)/2)",
        {
            "NIR": image.select("B8").divide(10000),
            "GREEN": image.select("B3").divide(10000),
            "RED": image.select("B4").divide(10000),
            "REDEDGE": image.select("B5").divide(10000),
        },
    ).rename("index")

sfdvi_img = sfdvi(image)
print("SFDVI mean over AOI:", index_mean(sfdvi_img))

## CIgreen

**Formula:** `CIgreen = (NIR / GREEN) - 1`

Chlorophyll Index Green. Ratio-based chlorophyll estimator strongly correlated with leaf/canopy chlorophyll; sensitive to early stress and nitrogen status.

*Legacy: `calculate_vegetation_index(image, "CIgreen")`; description from `vegetation_indices["CIgreen"]`.*

In [ ]:
def cigreen(image):
    # legacy: raw (unscaled) bands -- ratio is scale-invariant
    nir = image.select("B8")
    green = image.select("B3")
    return image.expression(
        "(NIR / GREEN) - 1", {"NIR": nir, "GREEN": green}
    ).rename("index")

cigreen_img = cigreen(image)
print("CIgreen mean over AOI:", index_mean(cigreen_img))

## NDRE

**Formula:** `NDRE = (NIR - REDEDGE) / (NIR + REDEDGE)`

Normalized Difference Red Edge. Uses the red-edge band; detects crop stress and nitrogen status earlier than NDVI and resists saturation in dense canopies.

*Legacy: `calculate_vegetation_index(image, "NDRE")`; description from `vegetation_indices["NDRE"]`.*

In [ ]:
def ndre(image):
    # legacy: image.normalizedDifference(["B8", "B5"]).rename("index")
    return image.normalizedDifference(["B8", "B5"]).rename("index")

ndre_img = ndre(image)
print("NDRE mean over AOI:", index_mean(ndre_img))

## ARVI

**Formula:** `ARVI = (NIR - (2*RED - BLUE)) / (NIR + (2*RED - BLUE))`

Atmospherically Resistant Vegetation Index. Uses the blue band to self-correct for aerosol scattering, so it is more robust than NDVI in hazy atmospheres.

*Legacy: `calculate_vegetation_index(image, "ARVI")`; description from `vegetation_indices["ARVI"]`.*

In [ ]:
def arvi(image):
    nir = image.select("B8").divide(10000)
    red = image.select("B4").divide(10000)
    blue = image.select("B2").divide(10000)
    return image.expression(
        "(NIR - (2 * RED - BLUE)) / (NIR + (2 * RED - BLUE))",
        {"NIR": nir, "RED": red, "BLUE": blue},
    ).rename("index")

arvi_img = arvi(image)
print("ARVI mean over AOI:", index_mean(arvi_img))

## NDMI

**Formula:** `NDMI = (NIR - SWIR1) / (NIR + SWIR1)`

Normalized Difference Moisture Index. NIR vs SWIR1 (B11); tracks canopy water content. Useful for drought monitoring, irrigation management, water-stress detection.

*Legacy: `calculate_vegetation_index(image, "NDMI")`; description from `vegetation_indices["NDMI"]`.*

In [ ]:
def ndmi(image):
    # legacy: image.normalizedDifference(["B8", "B11"]).rename("index")  (SWIR1 = B11)
    return image.normalizedDifference(["B8", "B11"]).rename("index")

ndmi_img = ndmi(image)
print("NDMI mean over AOI:", index_mean(ndmi_img))

## NBR

**Formula:** `NBR = (NIR - SWIR2) / (NIR + SWIR2)`

Normalized Burn Ratio. NIR vs SWIR2 (B12); maps burned areas and burn severity, and monitors post-fire vegetation recovery.

*Legacy: `calculate_vegetation_index(image, "NBR")`; description from `vegetation_indices["NBR"]`.*

In [ ]:
def nbr(image):
    # legacy: image.normalizedDifference(["B8", "B12"]).rename("index")  (SWIR2 = B12)
    return image.normalizedDifference(["B8", "B12"]).rename("index")

nbr_img = nbr(image)
print("NBR mean over AOI:", index_mean(nbr_img))

## SIPI

**Formula:** `SIPI = (NIR - BLUE) / (NIR - RED)`

Structure Insensitive Pigment Index. Assesses canopy stress while minimizing the influence of canopy structure (carotenoid-to-chlorophyll balance).

*Legacy: `calculate_vegetation_index(image, "SIPI")`; description from `vegetation_indices["SIPI"]`.*

In [ ]:
def sipi(image):
    nir = image.select("B8").divide(10000)
    red = image.select("B4").divide(10000)
    blue = image.select("B2").divide(10000)
    return image.expression(
        "(NIR - BLUE) / (NIR - RED)",
        {"NIR": nir, "RED": red, "BLUE": blue},
    ).rename("index")

sipi_img = sipi(image)
print("SIPI mean over AOI:", index_mean(sipi_img))

## NDWI

**Formula:** `NDWI = (GREEN - NIR) / (GREEN + NIR)`

Normalized Difference Water Index (McFeeters, green/NIR form). Sensitive to liquid water content in vegetation canopies and open water.

*Legacy: `calculate_vegetation_index(image, "NDWI")`; description from `vegetation_indices["NDWI"]`.*

In [ ]:
def ndwi(image):
    # legacy: image.normalizedDifference(["B3", "B8"]).rename("index")  -> (GREEN - NIR)/(GREEN + NIR)
    return image.normalizedDifference(["B3", "B8"]).rename("index")

ndwi_img = ndwi(image)
print("NDWI mean over AOI:", index_mean(ndwi_img))

## ReCI

**Formula:** `ReCI = (NIR / REDEDGE) - 1`

Red Edge Chlorophyll Index. Red-edge ratio estimator of chlorophyll; widely used in precision agriculture for crop health and nitrogen monitoring.

*Legacy: `calculate_vegetation_index(image, "ReCI")`; description from `vegetation_indices["ReCI"]`.*

In [ ]:
def reci(image):
    # legacy: raw (unscaled) bands; REDEDGE = B5
    nir = image.select("B8")
    rededge = image.select("B5")
    return image.expression(
        "(NIR / REDEDGE) - 1",
        {"NIR": nir, "REDEDGE": rededge},
    ).rename("index")

reci_img = reci(image)
print("ReCI mean over AOI:", index_mean(reci_img))

## MTCI

**Formula:** `MTCI = (NIR - REDEDGE) / (REDEDGE - RED)`

MERIS Terrestrial Chlorophyll Index. Red-edge slope index sensitive to canopy chlorophyll content; good for vegetation-health monitoring.

*Legacy: `calculate_vegetation_index(image, "MTCI")`; description from `vegetation_indices["MTCI"]`.*

In [ ]:
def mtci(image):
    # legacy: raw (unscaled) bands; REDEDGE = B5
    nir = image.select("B8")
    rededge = image.select("B5")
    red = image.select("B4")
    return image.expression(
        "(NIR - REDEDGE) / (REDEDGE - RED)",
        {"NIR": nir, "REDEDGE": rededge, "RED": red},
    ).rename("index")

mtci_img = mtci(image)
print("MTCI mean over AOI:", index_mean(mtci_img))

## MCARI

**Formula:** `MCARI = ((REDEDGE - RED) - 0.2 * (REDEDGE - GREEN)) * (REDEDGE / RED)`

Modified Chlorophyll Absorption Ratio Index. Sensitive to chlorophyll concentration and vegetation stress. (Legacy uses NIR/B8 in the red-edge term -- kept as-is.)

*Legacy: `calculate_vegetation_index(image, "MCARI")`; description from `vegetation_indices["MCARI"]`.*

In [ ]:
def mcari(image):
    # NOTE: legacy passes B8 (NIR) into the REDEDGE slot of this expression -- preserved faithfully.
    nir = image.select("B8").divide(10000)
    red = image.select("B4").divide(10000)
    green = image.select("B3").divide(10000)
    return image.expression(
        "((REDEDGE - RED) - 0.2 * (REDEDGE - GREEN)) * (REDEDGE / RED)",
        {"REDEDGE": nir, "RED": red, "GREEN": green},
    ).rename("index")

mcari_img = mcari(image)
print("MCARI mean over AOI:", index_mean(mcari_img))

## VARI

**Formula:** `VARI = (GREEN - RED) / (GREEN + RED - BLUE)`

Visible Atmospherically Resistant Index. Uses only visible bands and self-corrects for atmosphere; handy when NIR is unavailable or atmospheric correction is hard.

*Legacy: `calculate_vegetation_index(image, "VARI")`; description from `vegetation_indices["VARI"]`.*

In [ ]:
def vari(image):
    red = image.select("B4").divide(10000)
    green = image.select("B3").divide(10000)
    blue = image.select("B2").divide(10000)
    return image.expression(
        "(GREEN - RED) / (GREEN + RED - BLUE)",
        {"GREEN": green, "RED": red, "BLUE": blue},
    ).rename("index")

vari_img = vari(image)
print("VARI mean over AOI:", index_mean(vari_img))

## TVI

**Formula:** `TVI = 0.5 * (120 * (NIR - GREEN) - 200 * (RED - GREEN))`

Triangular Vegetation Index. Area of the triangle formed by green, red and NIR reflectance; a transform of NDVI sensitive to green biomass.

*Legacy: `calculate_vegetation_index(image, "TVI")`; description from `vegetation_indices["TVI"]`.*

In [ ]:
def tvi(image):
    nir = image.select("B8").divide(10000)
    red = image.select("B4").divide(10000)
    green = image.select("B3").divide(10000)
    return image.expression(
        "0.5 * (120 * (NIR - GREEN) - 200 * (RED - GREEN))",
        {"NIR": nir, "RED": red, "GREEN": green},
    ).rename("index")

tvi_img = tvi(image)
print("TVI mean over AOI:", index_mean(tvi_img))

In [ ]:
# Final mapping: index name -> the ee function that computes it (mirrors the legacy
# `index_functions` dict in RaviDialog.calculate_vegetation_index).
INDEX_FUNCTIONS = {
    "NDVI": ndvi,
    "GNDVI": gndvi,
    "EVI": evi,
    "EVI2": evi2,
    "SAVI": savi,
    "MSAVI": msavi,
    "SFDVI": sfdvi,
    "CIgreen": cigreen,
    "NDRE": ndre,
    "ARVI": arvi,
    "NDMI": ndmi,
    "NBR": nbr,
    "SIPI": sipi,
    "NDWI": ndwi,
    "ReCI": reci,
    "MTCI": mtci,
    "MCARI": mcari,
    "VARI": vari,
    "TVI": tvi,
}

def calculate_vegetation_index(image, index_name):
    """Compute a built-in vegetation index, returning an ee.Image with band 'index'.

    Mirrors legacy RaviDialog.calculate_vegetation_index.
    """
    if index_name not in INDEX_FUNCTIONS:
        raise ValueError(f"Unsupported vegetation index: {index_name}")
    return INDEX_FUNCTIONS[index_name](image)

# Verify: compute the AOI mean for every index and tabulate.
rows = []
for name, fn in INDEX_FUNCTIONS.items():
    rows.append({"index": name, "aoi_mean": index_mean(fn(image))})

summary = pd.DataFrame(rows).set_index("index")
print(summary)
summary